In [4]:
import requests
import matplotlib.pyplot as plt
import pandas as pd
import json

In [2]:
%pip install google-analytics-data pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.oauth2 import service_account

PROPERTY_ID = "321460044"

KEY_FILE = r"C:\Users\jlmow\Documents-C Drive\NSS-C Drive\Capstone\sfs-mrktg-76749b6efce7.json"

credentials = service_account.Credentials.from_service_account_file(
    KEY_FILE
)

client = BetaAnalyticsDataClient(credentials=credentials)

print("Connection setup completed.")

Connection setup completed.


In [4]:
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest
)
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[
        Dimension(name="date"),
        Dimension(name="sessionSourceMedium"),
        Dimension(name="pagePath")
    ],
    metrics=[
        Metric(name="sessions")
    ],
    date_ranges=[
        DateRange(
            start_date="730daysAgo",
            end_date="yesterday"
        )
    ],
 limit=100000
)
response = client.run_report(request)

print("Number of rows returned:", len(response.rows))

Number of rows returned: 27044


In [5]:
data = []

for row in response.rows:
    data.append({
        "date": row.dimension_values[0].value,
        "source_medium": row.dimension_values[1].value,
        "page_path": row.dimension_values[2].value,
        "sessions": row.metric_values[0].value
    })

ga4_df = pd.DataFrame(data)

ga4_df["date"] = pd.to_datetime(
    ga4_df["date"],
    format="%Y%m%d"
)

ga4_df["sessions"] = pd.to_numeric(
    ga4_df["sessions"]

)

ga4_df.sort_values(by="date")

,date,source_medium,page_path,sessions
9105,2024-08-03,(direct) / (none),/about-us/,1
9106,2024-08-03,(direct) / (none),/inforequest/,1
9122,2024-08-03,serchen.com / referral,/,1
9121,2024-08-03,hfm / video450,/client-services/site-inventory-database-build...,1
9120,2024-08-03,hfm / newslink,/insights/,1
...,...,...,...,...
9102,2026-08-02,(direct) / (none),/packages/service-optimizer/,2
9101,2026-08-02,(direct) / (none),/contact/,2
9100,2026-08-02,(direct) / (none),/client-services/,2
27027,2026-08-02,(direct) / (none),/client-services/workflow-setup/,1


In [6]:
ga4_df_date = ga4_df.groupby(['date','source_medium']).sum()
ga4_df_date

page_path  \
date       source_medium                                                                           
2024-08-03 (direct) / (none)                   //contact//about-us//inforequest//insights/evs...   
           bing / organic                                                                      /   
           google / organic                    //insights/hcahps-just-the-facts//packages/es-...   
           hfm / newslink                      /insights/evs-budgeting-know-your-numbers-lead...   
           hfm / video450                      /client-services/site-inventory-database-build...   
...                                                                                          ...   
2026-08-01 google / organic                     /packages/survey-optimizer//purchasing/services/   
2026-08-02 (direct) / (none)                   //packages/es-optimizer//support//client-servi...   
           bing / organic                                                /packages/es-optimizer/   
           google / organic                    /packages/survey-optimizer//insights/hcahps-ju...   
           nmhealth.sharepoint.com / referral                     /support/instructional-videos/   

                                               sessions  
date       source_medium                                 
2024-08-03 (direct) / (none)                         19  
           bing / organic                             1  
           google / organic                           4  
           hfm / newslink                             4  
           hfm / video450                             1  
...                                                 ...  
2026-08-01 google / organic                           3  
2026-08-02 (direct) / (none)                         33  
           bing / organic                             1  
           google / organic                          10  
           nmhealth.sharepoint.com / referral         1  

[5922 rows x 2 columns]

In [7]:
ga4_all = ga4_df.groupby(['date']).sum()
ga4_all

,source_medium,page_path,sessions
date,,,
2024-08-03,(direct) / (none)(direct) / (none)hfm / newsli...,//contact//insights/evs-budgeting-know-your-nu...,30
2024-08-04,bing / organic(direct) / (none)(direct) / (non...,/packages/es-optimizer///evs-visibility///insi...,13
2024-08-05,(direct) / (none)google / organic(direct) / (n...,///contact//support/instructional-videos//pack...,130
2024-08-06,(direct) / (none)(direct) / (none)(direct) / (...,//packages/es-optimizer//packages/survey-optim...,141
2024-08-07,(direct) / (none)google / organic(direct) / (n...,/packages/es-optimizer////support/instructiona...,91
...,...,...,...
2026-07-29,(direct) / (none)(direct) / (none)google / org...,//packages/es-optimizer//insights/copy-for-eng...,113
2026-07-30,(direct) / (none)linkedin.com / referralgoogle...,//insights/engineering-environmental-services/...,64
2026-07-31,(direct) / (none)google / organicgoogle / orga...,///packages/es-optimizer//packages/survey-opti...,86


In [8]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
from getpass import getpass

HUBSPOT_TOKEN = getpass("Paste your HubSpot access token: ")

Paste your HubSpot access token:  ········


In [13]:
url = "https://api.hubapi.com/crm/v3/objects/contacts"

headers = {
    "Authorization": f"Bearer {HUBSPOT_TOKEN}"
}

params = {
    "limit": 100,
    "properties": ",".join([
        "firstname",
        "lastname",
        "email",
        "phone",
        "jobtitle",
        "company",
        "lifecyclestage"
    ])
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

response.raise_for_status()

hubspot_data = response.json()

print("Number of contacts returned:", len(hubspot_data["results"]))

Number of contacts returned: 100


In [14]:
contacts = []

for contact in hubspot_data["results"]:
    properties = contact["properties"]

    contacts.append({
        "contact_id": contact["id"],
        "first_name": properties.get("firstname"),
        "last_name": properties.get("lastname"),
        "email": properties.get("email"),
        "phone": properties.get("phone"),
        "job_title": properties.get("jobtitle"),
        "company_name": properties.get("company"),
        "lifecycle_stage": properties.get("lifecyclestage"),
        "created_at": contact.get("createdAt"),
        "updated_at": contact.get("updatedAt")
    })

hubspot_contacts_df = pd.DataFrame(contacts)

hubspot_contacts_df

,contact_id,first_name,last_name,email,phone,job_title,company_name,lifecycle_stage,created_at,updated_at
0,101,C,Lucier,clucier@smartfacilitysoftware.com,3867474459,National Sales Director,SFS,opportunity,2021-03-31T14:39:21.872Z,2026-08-03T20:41:54.940Z
1,151,Shawn,Wright,swright@smartfacilitysoftware.com,(615)238-4829,President,None,opportunity,2021-03-31T14:47:25.623Z,2026-08-03T20:40:28.387Z
2,201,Terry,Vogt,terry.vogt@va.gov,(914) 737-4400,EMS-Manager,None,customer,2021-03-31T18:22:35.861Z,2023-07-17T14:51:15.822Z
3,202,Sherima,Felix,sgf9005@nyp.org,(212) 297-4593,Training and Development,None,45618968,2021-03-31T18:22:35.880Z,2022-12-28T16:37:51.190Z
4,203,Jerry,"Hiland, C.P.M.",jhiland@iupui.edu,(317) 274-5031,CFS Procurement Manager,None,customer,2021-03-31T18:22:35.879Z,2023-07-17T14:51:15.395Z
...,...,...,...,...,...,...,...,...,...,...
95,346,Shane,Matthews,smatthews@adena.org,(740) 779-8226,Systems Analyst,None,45618968,2021-03-31T18:22:39.770Z,2023-11-01T06:47:27.094Z
96,347,Bob,Scott,scott@willisconsultingservices.com,(610) 366-1870,Vice President of Business Services,None,lead,2021-03-31T18:22:39.885Z,2026-02-19T19:32:30.728Z
97,348,Jane,Volz Blackstone,mjvolz@medcentral.org,(419) 526-8500,Linen-EVS Secretary,None,45618968,2021-03-31T18:22:39.918Z,2023-11-01T07:37:02.597Z
98,349,"Jose ""Alex""",Burgos,burgosj@nyp.org,(212) 297-4593,Supervisor,None,45618968,2021-03-31T18:22:39.939Z,2022-12-28T16:40:56.034Z
